In [13]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from itertools import combinations
from collections import Counter
import warnings, gc, os
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED        = 2025
N_FOLDS     = 10
TARGET      = "Irrigation_Need"
target_map  = {"Low":0,"Medium":1,"High":2}
reverse_map = {0:"Low",1:"Medium",2:"High"}
COMP = "/home/chotu/Downloads/Irrigation_Submission/data/raw"
ORIG = "/home/chotu/Downloads/Irrigation_Submission/data/OG_data"
print("Setup done")

Setup done


In [14]:
train = pd.read_csv(COMP + "/train.csv")
test  = pd.read_csv(COMP + "/test.csv")
orig  = pd.read_csv(ORIG + "/irrigation_prediction.csv")

train[TARGET] = train[TARGET].map(target_map)
orig[TARGET]  = orig[TARGET].map(target_map)

CATS = ["Soil_Type","Crop_Type","Crop_Growth_Stage","Season",
        "Irrigation_Type","Water_Source","Mulching_Used","Region"]
NUMS = ["Soil_pH","Soil_Moisture","Organic_Carbon","Electrical_Conductivity",
        "Temperature_C","Humidity","Rainfall_mm","Sunlight_Hours",
        "Wind_Speed_kmh","Field_Area_hectare","Previous_Irrigation_mm"]

# ── Load soft probabilities from our 3 best models ──
# Upload soft_proba_stack.csv, soft_proba_nn.csv, soft_proba_pseudolabel.csv
# to Kaggle as a dataset called "irrigation-soft-probas"
SOFT_DIR = "/home/chotu/Downloads/Irrigation_Submission/data/irrigation-soft-probas"
sp_stack  = pd.read_csv(SOFT_DIR + "/soft_proba_stack.csv").sort_values("id").reset_index(drop=True)
sp_nn     = pd.read_csv(SOFT_DIR + "/soft_proba_nn.csv").sort_values("id").reset_index(drop=True)
sp_pseudo = pd.read_csv(SOFT_DIR + "/soft_proba_pseudolabel.csv").sort_values("id").reset_index(drop=True)

cols = ["proba_Low","proba_Medium","proba_High"]

# Ensemble soft probas: pseudo(50%) + stack(30%) + nn(20%)
soft_ensemble = (
    0.50 * sp_pseudo[cols].values +
    0.30 * sp_stack[cols].values  +
    0.20 * sp_nn[cols].values
)

test_sorted = test.sort_values("id").reset_index(drop=True)
test_sorted["soft_Low"]    = soft_ensemble[:, 0]
test_sorted["soft_Medium"] = soft_ensemble[:, 1]
test_sorted["soft_High"]   = soft_ensemble[:, 2]
test_sorted["pseudo_label"]= soft_ensemble.argmax(1)
test_sorted["confidence"]  = soft_ensemble.max(1)
test_sorted["pseudo_margin"] = np.sort(soft_ensemble, axis=1)[:,-1] - np.sort(soft_ensemble, axis=1)[:,-2]

# All test rows get pseudo-labels — weight by confidence
print(f"Test rows: {len(test_sorted):,}")
print(f"Pseudo confidence: mean={test_sorted['confidence'].mean():.4f}")
print(f"High confidence (>0.99): {(test_sorted['confidence']>0.99).sum():,}")
print(f"Uncertain (<0.80): {(test_sorted['confidence']<0.80).sum():,}")
print(f"Pseudo dist: {Counter(test_sorted['pseudo_label'].map(reverse_map))}")

Test rows: 270,000
Pseudo confidence: mean=0.9675
High confidence (>0.99): 7,446
Uncertain (<0.80): 6,543
Pseudo dist: Counter({'Low': 159894, 'Medium': 100110, 'High': 9996})


In [15]:
def engineer(df_list):
    for df in df_list:
        eps = 1e-6
        # Domain interactions
        df["water_avail"]    = df["Soil_Moisture"] + df["Rainfall_mm"] / 300.0
        df["heat_stress"]    = df["Temperature_C"] / (df["Soil_Moisture"] + eps)
        df["rain_cool"]      = df["Rainfall_mm"] / (df["Temperature_C"] + eps)
        df["wind_evap"]      = df["Wind_Speed_kmh"] * df["Temperature_C"] / 100.0
        df["soil_x_rain"]    = df["Soil_Moisture"] * df["Rainfall_mm"] / 1000.0
        df["temp_x_dry"]     = df["Temperature_C"] * (100 - df["Soil_Moisture"]) / 100.0
        df["ec_x_ph"]        = df["Electrical_Conductivity"] * df["Soil_pH"]
        df["sun_x_temp"]     = df["Sunlight_Hours"] * df["Temperature_C"]
        df["prev_ratio"]     = df["Previous_Irrigation_mm"] / (df["Rainfall_mm"] + eps)
        df["area_x_rain"]    = df["Field_Area_hectare"] * df["Rainfall_mm"]
        # Threshold flags
        df["soil_lt_25"]     = (df["Soil_Moisture"] < 25).astype(int)
        df["soil_lt_30"]     = (df["Soil_Moisture"] < 30).astype(int)
        df["temp_gt_30"]     = (df["Temperature_C"] > 30).astype(int)
        df["temp_gt_35"]     = (df["Temperature_C"] > 35).astype(int)
        df["rain_lt_300"]    = (df["Rainfall_mm"] < 300).astype(int)
        df["rain_lt_200"]    = (df["Rainfall_mm"] < 200).astype(int)
        df["wind_gt_10"]     = (df["Wind_Speed_kmh"] > 10).astype(int)
        df["wind_gt_15"]     = (df["Wind_Speed_kmh"] > 15).astype(int)
        # Proven logit priors
        df["logit_low"]      = (16.3173 - 11.0237*df["soil_lt_25"] - 5.8559*df["temp_gt_30"]
                                - 10.8500*df["rain_lt_300"] - 5.8284*df["wind_gt_10"])
        df["logit_med"]      = (4.6524 + 0.3290*df["soil_lt_25"] - 0.0204*df["temp_gt_30"]
                                + 0.1542*df["rain_lt_300"] + 0.0841*df["wind_gt_10"])
        df["logit_high"]     = (-20.9697 + 10.6947*df["soil_lt_25"] + 5.8763*df["temp_gt_30"]
                                + 10.6958*df["rain_lt_300"] + 5.7444*df["wind_gt_10"])
        # Digit features on key numerics
        for c in ["Soil_Moisture","Temperature_C","Rainfall_mm","Wind_Speed_kmh"]:
            for k in range(-1, 3):
                df[f"{c}_d{k}"] = (df[c] // (10**k) % 10).astype(int)
        # Cat combos
        df["crop_stage"]     = df["Crop_Type"].astype(str)+"_"+df["Crop_Growth_Stage"].astype(str)
        df["soil_region"]    = df["Soil_Type"].astype(str)+"_"+df["Region"].astype(str)
        df["season_irr"]     = df["Season"].astype(str)+"_"+df["Irrigation_Type"].astype(str)
        df["mulch_water"]    = df["Mulching_Used"].astype(str)+"_"+df["Water_Source"].astype(str)
        df["crop_season"]    = df["Crop_Type"].astype(str)+"_"+df["Season"].astype(str)
        df["soil_irr"]       = df["Soil_Type"].astype(str)+"_"+df["Irrigation_Type"].astype(str)
    return df_list

engineer([train, test_sorted, orig])
CAT_COMBOS = ["crop_stage","soil_region","season_irr","mulch_water","crop_season","soil_irr"]
ALL_CATS   = CATS + CAT_COMBOS

# 2-way combos for top cats
for c1, c2 in combinations(CATS[:5], 2):
    col = f"{c1}__{c2}"
    for df in [train, test_sorted, orig]:
        df[col] = df[c1].astype(str)+"_"+df[c2].astype(str)
    CAT_COMBOS.append(col)

ALL_CATS = CATS + CAT_COMBOS

# Label encode
for c in ALL_CATS:
    le = LabelEncoder()
    combined = pd.concat([train[c], test_sorted[c], orig[c]]).astype(str)
    le.fit(combined)
    for df in [train, test_sorted, orig]:
        df[c] = le.transform(df[c].astype(str))

# Orig-based TE
TE_FEATS = []
for c in CATS:
    for cls in [0,1,2]:
        col = f"TE_{c}_cls{cls}"
        te_map = orig.groupby(c)[TARGET].apply(lambda x:(x==cls).mean())
        for df in [train, test_sorted, orig]:
            df[col] = df[c].map(te_map).fillna(1/3)
        TE_FEATS.append(col)

# Freq encoding
FREQ_FEATS = []
for c in ALL_CATS:
    freq = pd.concat([train[c],orig[c],test_sorted[c]]).value_counts(normalize=True)
    for df in [train, test_sorted, orig]:
        df[f"FREQ_{c}"] = df[c].map(freq).fillna(0).astype("float32")
    FREQ_FEATS.append(f"FREQ_{c}")

# Soft proba features from ensemble (add them as features!)
SOFT_FEATS = ["soft_Low","soft_Medium","soft_High","confidence","pseudo_margin"]
for col in SOFT_FEATS:
    train[col]       = np.nan   # unknown for train
    orig[col]        = np.nan   # unknown for orig

NEW_NUMS = ["water_avail","heat_stress","rain_cool","wind_evap","soil_x_rain",
            "temp_x_dry","ec_x_ph","sun_x_temp","prev_ratio","area_x_rain",
            "soil_lt_25","soil_lt_30","temp_gt_30","temp_gt_35","rain_lt_300",
            "rain_lt_200","wind_gt_10","wind_gt_15","logit_low","logit_med","logit_high"]
DIG_FEATS = [c for c in train.columns if "_d-" in c or "_d0" in c or "_d1" in c or "_d2" in c]
ALL_FEATS = list(dict.fromkeys(NUMS + NEW_NUMS + ALL_CATS + TE_FEATS + FREQ_FEATS + DIG_FEATS))
print(f"Total features: {len(ALL_FEATS)}")

Total features: 120


In [16]:
X      = train[ALL_FEATS].values.astype(np.float32)
y      = train[TARGET].values
X_orig = orig[ALL_FEATS].values.astype(np.float32)
y_orig = orig[TARGET].values
X_te   = test_sorted[ALL_FEATS].values.astype(np.float32)

# Pseudo weights based on confidence
pseudo_y   = test_sorted["pseudo_label"].values
pseudo_conf = test_sorted["confidence"].values
# Weight = confidence * 0.8 (cap at 0.8 to not overpower real labels)
pseudo_w   = np.clip(pseudo_conf * 0.8, 0.1, 0.8)

print(f"Train: {X.shape}, Orig: {X_orig.shape}, Test: {X_te.shape}")
print(f"Pseudo weight stats: min={pseudo_w.min():.3f} mean={pseudo_w.mean():.3f} max={pseudo_w.max():.3f}")

Train: (630000, 120), Orig: (10000, 120), Test: (270000, 120)
Pseudo weight stats: min=0.398 mean=0.774 max=0.794


In [ ]:
lgb_params = {
    "objective":"multiclass","num_class":3,
    "metric":"multi_logloss","boosting_type":"gbdt",
    "n_estimators":3000,"learning_rate":0.01,
    "num_leaves":127,"max_depth":-1,
    "min_child_samples":15,"subsample":0.8,
    "colsample_bytree":0.7,"reg_alpha":0.1,
    "reg_lambda":1.0,"class_weight":"balanced",
    "random_state":SEED,"n_jobs":-1,"verbose":-1,
}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_lgb  = np.zeros((len(X),3))
test_lgb = np.zeros((len(X_te),3))

for fold,(tr_idx,val_idx) in enumerate(skf.split(X,y)):
    print(f"LGB Fold {fold+1}/{N_FOLDS}", end=" ")
    Xtr = np.vstack([X[tr_idx], X_orig, X_te])
    ytr = np.concatenate([y[tr_idx], y_orig, pseudo_y])
    sw  = np.concatenate([
        compute_sample_weight("balanced", y[tr_idx]),
        compute_sample_weight("balanced", y_orig),
        pseudo_w
    ])
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(Xtr, ytr, sample_weight=sw,
              eval_set=[(X[val_idx],y[val_idx])],
              callbacks=[lgb.early_stopping(100,verbose=False),lgb.log_evaluation(-1)])
    oof_lgb[val_idx]  = model.predict_proba(X[val_idx])
    test_lgb         += model.predict_proba(X_te) / N_FOLDS
    score = balanced_accuracy_score(y[val_idx], oof_lgb[val_idx].argmax(1))
    print(f"| BA={score:.5f}")
    del model; gc.collect()

lgb_oof = balanced_accuracy_score(y, oof_lgb.argmax(1))
print(f"LGB OOF: {lgb_oof:.5f}")

LGB Fold 1/10 | BA=0.97253
LGB Fold 2/10 | BA=0.97010
LGB Fold 3/10 | BA=0.97386
LGB Fold 4/10 | BA=0.96980
LGB Fold 5/10 | BA=0.96808
LGB Fold 6/10 | BA=0.97097
LGB Fold 7/10 | BA=0.97402
LGB Fold 8/10 | BA=0.97118
LGB Fold 9/10 | BA=0.97260
LGB Fold 10/10 | BA=0.97549
LGB OOF: 0.97186


In [ ]:
cb_params = dict(
    iterations=2000, learning_rate=0.01, depth=8,
    loss_function="MultiClass", eval_metric="Accuracy",
    class_weights=[1.0,1.0,8.0],
    random_seed=SEED, verbose=0, task_type="GPU",
    early_stopping_rounds=100,
    border_count=64, bootstrap_type="Bernoulli", subsample=0.8,
)

oof_cat  = np.zeros((len(X),3))
test_cat = np.zeros((len(X_te),3))

for fold,(tr_idx,val_idx) in enumerate(skf.split(X,y)):
    print(f"CAT Fold {fold+1}/{N_FOLDS}", end=" ")
    Xtr = np.vstack([X[tr_idx], X_orig, X_te])
    ytr = np.concatenate([y[tr_idx], y_orig, pseudo_y])
    sw  = np.concatenate([
        compute_sample_weight("balanced", y[tr_idx]),
        compute_sample_weight("balanced", y_orig),
        pseudo_w
    ])
    model = cb.CatBoostClassifier(**cb_params)
    model.fit(Xtr, ytr, sample_weight=sw,
              eval_set=(X[val_idx],y[val_idx]),
              use_best_model=True)
    oof_cat[val_idx]  = model.predict_proba(X[val_idx])
    test_cat         += model.predict_proba(X_te) / N_FOLDS
    score = balanced_accuracy_score(y[val_idx], oof_cat[val_idx].argmax(1))
    print(f"| BA={score:.5f}")
    del model; gc.collect()

cat_oof = balanced_accuracy_score(y, oof_cat.argmax(1))
print(f"CAT OOF: {cat_oof:.5f}")

CAT Fold 1/10 | BA=0.97290
CAT Fold 2/10 | BA=0.97026
CAT Fold 3/10 | BA=0.97424
CAT Fold 4/10 | BA=0.97119
CAT Fold 5/10 | BA=0.96771
CAT Fold 6/10 | BA=0.97156
CAT Fold 7/10 | BA=0.97300
CAT Fold 8/10 | BA=0.97257
CAT Fold 9/10 | BA=0.97325
CAT Fold 10/10 | BA=0.97403
CAT OOF: 0.97207


In [ ]:
xgb_params = dict(
    n_estimators=3000, learning_rate=0.01,
    max_depth=7, subsample=0.8, colsample_bytree=0.7,
    objective="multi:softprob", num_class=3,
    eval_metric="mlogloss", random_state=SEED,
    device="cuda", n_jobs=-1,
    alpha=1, reg_lambda=2,
)

oof_xgb  = np.zeros((len(X),3))
test_xgb = np.zeros((len(X_te),3))

for fold,(tr_idx,val_idx) in enumerate(skf.split(X,y)):
    print(f"XGB Fold {fold+1}/{N_FOLDS}", end=" ")
    Xtr = np.vstack([X[tr_idx], X_orig, X_te])
    ytr = np.concatenate([y[tr_idx], y_orig, pseudo_y])
    sw  = np.concatenate([
        compute_sample_weight("balanced", y[tr_idx]),
        compute_sample_weight("balanced", y_orig),
        pseudo_w
    ])
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(Xtr, ytr, sample_weight=sw,
              eval_set=[(X[val_idx],y[val_idx])],
              verbose=False)
    oof_xgb[val_idx]  = model.predict_proba(X[val_idx])
    test_xgb         += model.predict_proba(X_te) / N_FOLDS
    score = balanced_accuracy_score(y[val_idx], oof_xgb[val_idx].argmax(1))
    print(f"| BA={score:.5f}")
    del model; gc.collect()

xgb_oof = balanced_accuracy_score(y, oof_xgb.argmax(1))
print(f"XGB OOF: {xgb_oof:.5f}")

XGB Fold 1/10 | BA=0.97070
XGB Fold 2/10 | BA=0.96750
XGB Fold 3/10 | BA=0.97038
XGB Fold 4/10 | BA=0.96570
XGB Fold 5/10 | BA=0.96293
XGB Fold 6/10 | BA=0.96889
XGB Fold 7/10 | BA=0.97333
XGB Fold 8/10 | BA=0.96871
XGB Fold 9/10 | BA=0.96965
XGB Fold 10/10 | BA=0.97349
XGB OOF: 0.96913


In [23]:
# Average all 3 models
avg_probs = (test_lgb + test_cat + test_xgb) / 3
oof_avg   = (oof_lgb  + oof_cat  + oof_xgb)  / 3
avg_score = balanced_accuracy_score(y, oof_avg.argmax(1))
print(f"3-model avg OOF: {avg_score:.5f}")

# Optuna class weight tuning on OOF
def cw_obj(trial):
    cw = np.array([trial.suggest_float(f"cw{i}",0.3,3.0) for i in range(3)])
    adj = oof_avg * cw
    adj /= adj.sum(axis=1, keepdims=True)
    return balanced_accuracy_score(y, adj.argmax(1))

study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
study.optimize(cw_obj, n_trials=1000, show_progress_bar=True)
best_cw = np.array([study.best_params[f"cw{i}"] for i in range(3)])
print(f"Best class weights: Low={best_cw[0]:.4f} Med={best_cw[1]:.4f} High={best_cw[2]:.4f}")

# Apply to test
final_probs = avg_probs * best_cw
final_probs /= final_probs.sum(axis=1, keepdims=True)

oof_tuned = oof_avg * best_cw
oof_tuned /= oof_tuned.sum(axis=1, keepdims=True)
tuned_score = balanced_accuracy_score(y, oof_tuned.argmax(1))
print(f"After CW tuning OOF: {tuned_score:.5f}")

# Blend new model with existing soft probas (V2 as anchor)
v2 = pd.read_csv("/home/chotu/Downloads/Irrigation_Submission/0.98151V2.csv")
v2_labels = v2.sort_values("id")["Irrigation_Need"].values
v2_soft = np.zeros((len(v2_labels),3))
for i,l in enumerate(v2_labels):
    idx = ["Low","Medium","High"].index(l)
    v2_soft[i,idx] = 0.97
    for j in range(3):
        if j!=idx: v2_soft[i,j]=0.015

# Final blend: new model(60%) + V2(40%)
mega_blend = 0.60*final_probs + 0.40*v2_soft
mega_preds = [reverse_map[p] for p in mega_blend.argmax(1)]

# Save
sub = pd.read_csv(COMP+"/sample_submission.csv")
test_ids = test_sorted["id"].values
sub_out = pd.DataFrame({"id":test_ids, TARGET:mega_preds})
sub_out = sub_out.sort_values("id").reset_index(drop=True)
sub_out.to_csv("submission_FINAL.csv", index=False)
print("submission_FINAL.csv saved!")
print(sub_out[TARGET].value_counts().to_dict())

# Soft probas for further ensembling
sp_out = pd.DataFrame(final_probs, columns=["proba_Low","proba_Medium","proba_High"])
sp_out.insert(0,"id",test_ids)
sp_out.to_csv("soft_proba_FINAL.csv", index=False)
print("soft_proba_FINAL.csv saved!")

3-model avg OOF: 0.97143


  0%|          | 0/1000 [00:00<?, ?it/s]

Best class weights: Low=0.3345 Med=1.0001 High=2.4507
After CW tuning OOF: 0.97362
submission_FINAL.csv saved!
{'Low': 159633, 'Medium': 100026, 'High': 10341}
soft_proba_FINAL.csv saved!
